# Park Access in Lafayette

**Series:** Case study: Park Access (Python) (notebook 1 of 1)
**Time:** about 60 minutes

**In this notebook you will:**
- Turn a policy question into a measurable definition of access
- Decide which features count as parks, and test how much that decision matters
- Estimate who lives within a 10-minute walk of a park: residents, children and car-free households
- Find the neighborhoods with the least access and describe them
- Write down the limitations of a buffer-based access measure

**You should already know:** the Geoprocessing in Python notebooks (<a href="https://colab.research.google.com/github/SpatialTurn/DataAnalysis/blob/main/notebooks/01a_Buffers_and_Selecting_by_Location.ipynb">01a</a> to <a href="https://colab.research.google.com/github/SpatialTurn/DataAnalysis/blob/main/notebooks/01c_Distance_and_Nearest_Features.ipynb">01c</a>) in the Data Analysis module.

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## The question

The **10-minute walk** standard, used by many cities and by the Trust for Public Land's ParkScore, asks that
every resident live within about a half mile (**800 m**) of a park. We ask three questions for Lafayette and
West Lafayette, Indiana:

1. What share of residents lives within 800 m of a public park?
2. Are **children** and **households without a car**, who depend most on nearby parks, as well served as everyone
   else?
3. Which neighborhoods have the least access?

The method is the one from the Geoprocessing episode: buffer, dissolve, clip, intersect and areal weighting.
The new work is in the **decisions**: what counts as a park, how far is "close", and how much the answer depends on
those choices.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

tracts = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_tracts_acs.geojson").to_crs(epsg=26916)
parks = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_parks.geojson").to_crs(epsg=26916)
city = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/study_area.geojson").to_crs(epsg=26916)
print(len(tracts), "tracts,", len(parks), "parks from OpenStreetMap")

## Step 1: What counts as a park?

The parks layer holds every feature tagged `leisure=park` in OpenStreetMap. Read the list: some are not parks a
family could walk to and use freely.

In [ ]:
print(", ".join(sorted(parks["name"])))

Campus lawns and plazas (the malls, Bell Tower Area, Spitzer Court), intramural fields, a scout camp, a shrine, an
apartment complex's dog park and Wolf Park (a wolf research center with admission fees) are tagged as parks but
are not **public recreational parks** for the question we are asking. We keep two versions so we can test the
decision later.

In [ ]:
NOT_PUBLIC_PARKS = [
    "Bell Tower Area", "Centennial Mall", "Centennial Mall South", "Memorial Mall", "Purdue Mall",
    "Stadium Mall", "De Bedout Plaza", "Spitzer Court", "Purdue Gold Intramural Fields",
    "Cary Camp, Sagamore Council, BSA", "Our Lady of Lourdes Shrine", "West Ridge Apartments Dog Park",
    "Wolf Park",
]
public = parks[~parks["name"].isin(NOT_PUBLIC_PARKS)]
print(len(public), "public parks kept,", len(parks) - len(public), "removed")

Write such decisions down, with the reason. Another analyst (or you, next year) must be able to repeat them, and a
reader must be able to disagree with them.

## Step 2: The 10-minute walk zone

Buffer each park by 800 m **from its edge** (a polygon buffer grows outward from the boundary), then dissolve the
overlapping buffers into one zone.

In [ ]:
WALK = 800                       # meters, about 10 minutes at an average walking pace
zone = gpd.GeoDataFrame(geometry=[public.buffer(WALK).union_all()], crs=26916)

ax = city.plot(color="whitesmoke", edgecolor="black", linewidth=1, figsize=(8, 8))
zone.clip(city).plot(ax=ax, color="#a1d99b", alpha=0.6)
public.plot(ax=ax, color="#31a354")
xmin, ymin, xmax, ymax = city.total_bounds
ax.set_xlim(xmin - 500, xmax + 500); ax.set_ylim(ymin - 500, ymax + 500)
ax.set_title("Within 800 m of a public park (light green)")
ax.set_axis_off()
plt.show()

## Step 3: Who lives in the zone?

As in Geoprocessing notebook 01b: clip the tracts to the city, keep each tract's original area, intersect with the
zone, and split each tract's counts by area share (areal weighting). We do it for three groups at once.

In [ ]:
GROUPS = {"total_pop": "Residents", "pop_under18": "Children (under 18)", "hh_no_vehicle": "Households without a car"}

tracts["tract_m2"] = tracts.area
in_city = gpd.clip(tracts, city)
share_city = in_city.area / in_city["tract_m2"]


def access(parks_layer, walk=WALK):
    """Share of each group living within `walk` meters of the parks, by areal weighting."""
    z = gpd.GeoDataFrame(geometry=[parks_layer.buffer(walk).union_all()], crs=26916)
    near = gpd.overlay(in_city, z, how="intersection")
    share_near = near.area / near["tract_m2"]
    return {label: (near[col] * share_near).sum() / (in_city[col] * share_city).sum() * 100
            for col, label in GROUPS.items()}


results = pd.Series(access(public)).round(1)
results

About seven in ten residents live within 800 m of a public park, but **children are noticeably less well
served** than residents as a whole, while households without a car are better served. A finding like the
children's gap is the kind of result a city can act on, so it deserves the checks in the next steps.

## Step 4: Access by neighborhood

In [ ]:
z = gpd.GeoDataFrame(geometry=[public.buffer(WALK).union_all()], crs=26916)
near = gpd.overlay(in_city, z, how="intersection")
near["pop_near"] = near.area / near["tract_m2"] * near["total_pop"]

by_tract = in_city.assign(pop_city=share_city * in_city["total_pop"]).dissolve(
    by="GEOID", aggfunc={"name": "first", "pop_city": "sum", "pop_under18": "first", "total_pop": "first",
                         "median_income": "first", "households": "first", "hh_no_vehicle": "first"})
by_tract["pop_near"] = near.groupby("GEOID")["pop_near"].sum()
by_tract["pop_near"] = by_tract["pop_near"].fillna(0)
by_tract["access_pct"] = by_tract["pop_near"] / by_tract["pop_city"] * 100

ax = by_tract.plot(column="access_pct", cmap="Greens", scheme="UserDefined",
                   classification_kwds={"bins": [25, 50, 75, 100]}, edgecolor="#969696", linewidth=0.4,
                   legend=True, legend_kwds={"title": "Residents within\n800 m of a park (%)", "loc": "lower left",
                                             "fmt": "{:.0f}"}, figsize=(8, 8))
public.plot(ax=ax, facecolor="none", edgecolor="#252525", hatch="....", linewidth=0.6)   # parks: hatched outlines
city.boundary.plot(ax=ax, color="black", linewidth=1, linestyle="--")
ax.set_xlim(xmin - 500, xmax + 500); ax.set_ylim(ymin - 500, ymax + 500)
ax.set_title("Share of residents within 800 m of a public park (parks hatched)", loc="left")
ax.set_axis_off()
plt.show()

In [ ]:
low = by_tract[by_tract["pop_city"] > 1000].sort_values("access_pct").head(5)
low[["name", "pop_city", "access_pct", "median_income"]].round(0)

Census Tracts 17.01 (east-central Lafayette) and 15.01 (southwest Lafayette), each with thousands of residents
and median incomes well below the county's typical level, have almost no public park within 800 m. These are the places to look at first, on a map
and on the ground.

## Step 5: Is access fair?

Compare access across income groups (thirds of tracts by median income), weighting by population.

In [ ]:
by_tract["income_group"] = pd.qcut(by_tract["median_income"], 3, labels=["Lower third", "Middle third", "Upper third"])
equity = by_tract.groupby("income_group", observed=True).apply(
    lambda d: d["pop_near"].sum() / d["pop_city"].sum() * 100)
print(equity.round(1))
print(by_tract["median_income"].isna().sum(), "tracts without an income estimate are left out of the groups")

There is no simple income gradient here: the middle third is best served, and the lower and upper thirds are
similar. That is a finding too. "No clear pattern" is worth reporting, as long as you also report the places
where access is lowest (Step 4).

## Step 6: How much do the decisions matter?

A **sensitivity analysis** reruns the analysis with different, defensible choices.

In [ ]:
table = pd.DataFrame({
    "Public parks, 800 m": access(public, 800),
    "All OSM parks, 800 m": access(parks, 800),
    "Public parks, 400 m": access(public, 400),
}).round(1)
table

Dropping campus lawns and private spaces barely changes the 800 m result (they sit where public parks already
are), but the **distance** matters a lot: at 400 m (a 5-minute walk) fewer than half of residents are served, and
the children's gap remains. Report the threshold, its source, and how the answer changes.

## Limitations

- **Straight lines.** A buffer ignores barriers: rivers, railroads and highways (the Wabash runs between the two
  cities). Network distance along walkable streets, as in the Data Analysis module's Network Analysis episode, is
  more realistic.
- **Edges, not entrances.** Large fenced parks are entered at a few gates.
- **All parks are equal.** A pocket park and a 100 ha park with pools and trails count the same. Weighting by size or
  facilities changes the picture.
- **Areal weighting** assumes people are spread evenly within tracts (see the Raster Analysis episode for a
  dasymetric alternative).
- **OpenStreetMap completeness.** A park missing from OSM looks like a gap in access. Check the low-access tracts
  against the cities' official park lists before drawing conclusions.

## Try it yourself

Many small parks offer little. Redo the main result (Step 3) using only public parks of **at least 2 hectares**.
How do the shares for residents and children change?

*Hint:* Filter `public` by `public.area`, then call `access()`.

<details>
<summary><b>Show a solution</b></summary>

```python
big_public = public[public.area >= 20000]          # 2 ha = 20,000 m2
print(len(big_public), "parks of 2 ha or more")
pd.Series(access(big_public)).round(1)
```

</details>

In [ ]:
# Write your answer here

## Recap

- Access measures start with definitions: what counts as a park, and how far is close
- Buffer from park edges, dissolve, intersect with tracts and areal-weight counts for each group
- Compare groups (residents, children, car-free households) and places, not just one city-wide number
- Test the decisions with a sensitivity table, and state the limitations

**Next:** the QGIS version of this case study, or the Food Deserts case study, which adds Census API data and
the USDA's own access measures.